# Burmese Agriculture CNER: Hyperparameter Grid Search Sweep Tool

This notebook is a specialized companion to `my_agri_cner_v2.ipynb`. Instead of running slow 5-fold cross-validation over 30 epochs with fixed params, this notebook is designed for **rapid hyperparameter exploration**.

### ⚡ Design Principles:
1. **Fold 0 Sweep**: Runs experiments exclusively on **Fold 0** of the best 2 configurations (`bioes_word` and `bioes_syllable` with embeddings) to preserve GPU allocation.
2. **Short Epoch Sweep**: Trains each combination for **only 1 epoch** (or a custom low count) to test convergence speed and find out what works best in under a minute per run.
3. **Dynamic Swapping**: Overrides network dimensions, batch sizes, optimizers, learning rates, and dropout rates programmatically.
4. **Ranked Sweeps & Bar Charts**: Compiles a real-time leaderboard ranking combinations from best-to-worst based on F1-score, and renders interactive performance horizontal bar charts.


### Step 1: Connect to Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


### Step 2: Configure Environment and Paths


In [ ]:
import os
import sys
import shutil

# Project directory in Google Drive
%env PJ_DIR=/content/drive/My Drive/MyAgriNER_copy
project_root = '/NCRFpp'

print(f'Project execution root is: {project_root}')


### Step 3: Clone NCRFpp Repository & Install Dependencies


In [ ]:
if not os.path.exists('/NCRFpp'):
    !git clone https://github.com/jiesutd/NCRFpp.git /NCRFpp
else:
    print('NCRFpp repository already exists.')

# Install the necessary libraries
with open('requirements.txt', 'w') as f:
    f.write('torch\nnumpy\n')
!pip install -r requirements.txt


### Step 4: Persistent Google Drive Symlinking & Dataset Extraction


In [ ]:
import os
import shutil

# 1. Define persistent directories in Google Drive
drive_root = '/content/drive/My Drive/MyAgriNER'
drive_data = os.path.join(drive_root, 'data')
drive_models = os.path.join(drive_root, 'models')
drive_output = os.path.join(drive_root, 'output')

# Create directories on Google Drive if they don't exist
os.makedirs(drive_data, exist_ok=True)
os.makedirs(drive_models, exist_ok=True)
os.makedirs(drive_output, exist_ok=True)

# 2. Extract data.zip directly to Google Drive (only if bio_word folder or emb files are missing to save time!)
already_extracted = os.path.exists(os.path.join(drive_data, 'bio_word')) and any(f.endswith('.emb') for f in os.listdir(drive_data)) if os.path.exists(drive_data) else False

if not already_extracted:
    zip_paths = [
        "/content/data.zip",
        "/content/drive/My Drive/MyAgriNER/data.zip",
        "/content/drive/My Drive/data.zip"
    ]
    found_zip = None
    for p in zip_paths:
        if os.path.exists(p):
            found_zip = p
            break
            
    if found_zip:
        print(f"Found data.zip at: {found_zip}")
        print("Extracting data.zip directly to Google Drive... This may take a minute.")
        !unzip -q -o "{found_zip}" -d "/content/drive/My Drive/MyAgriNER/"
        print("Extraction complete!")
    else:
        print("Warning: data.zip not found! If you have not uploaded pre-split folders, please upload data.zip to Google Drive (MyAgriNER/) or Colab (/content/).")
else:
    print("Data directory already exists and contains extracted files on Google Drive. Skipping extraction.")

# 3. Symlink /NCRFpp directories to Google Drive
for folder, drive_path in [('data', drive_data), ('models', drive_models), ('output', drive_output)]:
    local_path = f'/NCRFpp/{folder}'
    if os.path.exists(local_path):
        if os.path.islink(local_path):
            os.unlink(local_path)
        else:
            shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)
    print(f'Symlinked local {local_path} -> persistent Google Drive: {drive_path}')


### Step 5: Robust K-Fold Dataset Splitter (Rotating Blocks)
Instead of random shuffling (which breaks cross-validation sequence standards), this cell splits each standard CoNLL file into 5 fold directories under `data/{setup_name}/fold_{fold_idx}/` containing standard `train.conll`, `dev.conll`, and `test.conll` files, respecting sentence boundaries.

**Smart Skipping**: It automatically checks your Google Drive, and if the split folders are already generated, it skips this process to save time.


In [ ]:
import os
from pathlib import Path

def load_conll_sentences(file_path):
    """Parses a CoNLL file and returns a list of raw sentence blocks."""
    sentences = []
    current_sentence = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            current_sentence.append(line)
            if not line.strip():
                if current_sentence:
                    sentences.append("".join(current_sentence))
                    current_sentence = []
        if current_sentence:
            content = "".join(current_sentence)
            if not content.endswith('\n'):
                content += '\n'
            if not content.endswith('\n\n'):
                content += '\n'
            sentences.append(content)
    return sentences

def split_sentences_by_ratio(sentences, ratio, folds):
    """Splits sentences into folds using rotating block selection."""
    B = sum(ratio)
    N = len(sentences)
    X, Y, Z = ratio
    
    folds_data = []
    for i in range(folds):
        S_i = int(i * B / folds)
        
        # Rotating block indices modulo B
        train_blocks = [(S_i + b) % B for b in range(X)]
        dev_blocks = [(S_i + X + b) % B for b in range(Y)]
        test_blocks = [(S_i + X + Y + b) % B for b in range(Z)]
        
        def get_sentences_for_blocks(blocks):
            selected = []
            for b in sorted(blocks):
                start_idx = int(b * N / B)
                end_idx = int((b + 1) * N / B)
                selected.extend(sentences[start_idx:end_idx])
            return selected
            
        train_sents = get_sentences_for_blocks(train_blocks)
        dev_sents = get_sentences_for_blocks(dev_blocks)
        test_sents = get_sentences_for_blocks(test_blocks)
        
        folds_data.append((train_sents, dev_sents, test_sents))
    return folds_data

def process_file(file_path, output_dir, ratio, folds):
    print(f"Processing CoNLL file: '{file_path}'")
    sentences = load_conll_sentences(file_path)
    print(f"  Loaded {len(sentences)} sentences.")
    
    folds_data = split_sentences_by_ratio(sentences, ratio, folds)
    base_name = Path(file_path).stem
    
    setup_dir = Path(output_dir) / base_name
    setup_dir.mkdir(parents=True, exist_ok=True)
    
    for i, (train, dev, test) in enumerate(folds_data):
        fold_dir = setup_dir / f"fold_{i}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        
        with open(fold_dir / "train.conll", 'w', encoding='utf-8') as f:
            f.writelines(train)
        with open(fold_dir / "dev.conll", 'w', encoding='utf-8') as f:
            f.writelines(dev)
        with open(fold_dir / "test.conll", 'w', encoding='utf-8') as f:
            f.writelines(test)
            
        print(f"    Fold {i} -> Train: {len(train)} sents, Dev: {len(dev)} sents, Test: {len(test)} sents")

data_dir = '/NCRFpp/data'

# Check if splits are already present on Google Drive
has_splits = False
if os.path.exists(data_dir):
    subdirs = [os.path.join(data_dir, d) for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    for subdir in subdirs:
        if 'fold_0' in os.listdir(subdir):
            has_splits = True
            break

if not has_splits:
    conll_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir)
                   if f.endswith('.conll') and not f.startswith(('train.', 'dev.', 'test.'))]
    for file in sorted(conll_files):
        process_file(file, data_dir, [8, 1, 1], 5)
    print('\nAll datasets split into 5 folds successfully!')
else:
    print('Dataset splits already exist on Google Drive. Skipping splitting phase!')


### Step 6: Dynamic Training & Decoding Configuration Generator
This cell generates `.train.config` and `.decode.config` files for all configurations, including variations with and without fastText pre-trained embeddings.


In [ ]:
project_root = '/NCRFpp'

def get_emb_path(name):
    """Finds the most specific pretrained embedding file, with fallbacks."""
    candidates = [
        f"{project_root}/data/custom_burmese_agri_word.emb",
        f"{project_root}/data/burmese_agri_word.emb",
        f"{project_root}/data/custom_burmese_agri_syllable.emb",
        f"{project_root}/data/burmese_agri_syllable.emb",
    ]
    is_word = "word" in name.lower()
    is_syllable = "syllable" in name.lower()

    if is_word:
        for p in candidates:
            if "word" in p and os.path.exists(p): return p
        return f"{project_root}/data/burmese_agri_word.emb"
    elif is_syllable:
        for p in candidates:
            if "syllable" in p and os.path.exists(p): return p
        return f"{project_root}/data/burmese_agri_syllable.emb"
    return None

def create_sweep_config(setup_name, fold=0, with_emb=True, params=None, config_name=None, epochs=1):
    """Generates a train config programmatically with hyperparameter overrides."""
    word_emb_setting = ""
    if with_emb:
        emb_path = get_emb_path(setup_name)
        if emb_path:
            word_emb_setting = f"word_emb_dir={emb_path}"

    # Default parameters
    p = {
        "optimizer": "ADAM",
        "learning_rate": 0.001,
        "dropout": 0.3,
        "hidden_dim": 200,
        "batch_size": 10,
        "ave_batch_loss": "False"
    }
    if params:
        p.update(params)

    train_content = f"""
### I/O ###
train_dir={project_root}/data/{setup_name}/fold_{fold}/train.conll
dev_dir={project_root}/data/{setup_name}/fold_{fold}/dev.conll
test_dir={project_root}/data/{setup_name}/fold_{fold}/test.conll
model_dir={project_root}/models/{config_name}
{word_emb_setting}

norm_word_emb=False
norm_char_emb=False
number_normalized=True
seg=True
word_emb_dim=200
char_emb_dim=200

###NetworkConfiguration###
use_crf=True
use_char=True
word_seq_feature=LSTM
char_seq_feature=CNN

###TrainingSetting###
status=train
optimizer={p['optimizer']}
iteration={epochs}
batch_size={p['batch_size']}
ave_batch_loss={p['ave_batch_loss']}

###Hyperparameters###
cnn_layer=4
char_hidden_dim=50
hidden_dim={p['hidden_dim']}
dropout={p['dropout']}
lstm_layer=1
bilstm=True
learning_rate={p['learning_rate']}
lr_decay=0
momentum=0
l2=1e-8
gpu
clip=5.0
"""
    train_config_path = f"{project_root}/{config_name}.train.config"
    with open(train_config_path, 'w') as f:
        f.write(train_content.strip())

def create_fold_decode_config(config_name, setup_name, fold, model_path, dset_path):
    """Generates the .decode.config configuration file for evaluation."""
    decode_content = f"""
### I/O ###
status=decode
raw_dir={project_root}/data/{setup_name}/fold_{fold}/test.conll
decode_dir={project_root}/output/{config_name}.test.out
dset_dir={dset_path}
load_model_dir={model_path}

nbest=1
gpu
"""
    decode_config_path = f"{project_root}/{config_name}.decode.config"
    with open(decode_config_path, 'w') as f:
        f.write(decode_content.strip())

print("Dynamic sweep configuration generator compiled successfully!")


### Step 7: Standard CoNLL Entity-Level Performance Evaluator
We implement the official CoNLL entity-level metric (exact matches of type, start, and end token indices) in pure python. This avoids dependencies and ensures complete reliability of computed precision, recall, and F1 scores.


In [ ]:
def get_entities(tags):
    """Extracts standard entity triples (type, start_idx, end_idx) from tags list."""
    entities = []
    current_entity = None
    
    for i, tag in enumerate(tags):
        if tag == "O" or tag == "<pad>" or tag == "<unk>":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue
            
        if "-" in tag:
            boundary, entity_type = tag.split("-", 1)
        else:
            boundary = tag
            entity_type = "ENTITY"
            
        boundary = boundary.upper()
        
        if boundary == "B":
            if current_entity:
                entities.append(current_entity)
            current_entity = {"type": entity_type, "start": i, "end": i}
        elif boundary == "I":
            if current_entity and current_entity["type"] == entity_type:
                current_entity["end"] = i
            else:
                if current_entity:
                    entities.append(current_entity)
                current_entity = {"type": entity_type, "start": i, "end": i}
        elif boundary == "E":
            if current_entity and current_entity["type"] == entity_type:
                current_entity["end"] = i
                entities.append(current_entity)
                current_entity = None
            else:
                if current_entity:
                    entities.append(current_entity)
                entities.append({"type": entity_type, "start": i, "end": i})
                current_entity = None
        elif boundary == "S":
            if current_entity:
                entities.append(current_entity)
            entities.append({"type": entity_type, "start": i, "end": i})
            current_entity = None
            
    if current_entity:
        entities.append(current_entity)
        
    return set((ent["type"], ent["start"], ent["end"]) for ent in entities)

def evaluate_predictions(file_path):
    """Parses an NCRFpp predictions output file and computes entity metrics."""
    gold_tags = []
    pred_tags = []
    current_gold = []
    current_pred = []
    
    if not os.path.exists(file_path):
        return None
        
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if current_gold:
                    gold_tags.append(current_gold)
                    pred_tags.append(current_pred)
                    current_gold = []
                    current_pred = []
                continue
            parts = line.split()
            if len(parts) >= 3:
                current_gold.append(parts[-2])
                current_pred.append(parts[-1])
            elif len(parts) == 2:
                current_gold.append(parts[0])
                current_pred.append(parts[1])
                
        if current_gold:
            gold_tags.append(current_gold)
            pred_tags.append(current_pred)
            
    total_gold_entities = 0
    total_pred_entities = 0
    correct_entities = 0
    
    for g_seq, p_seq in zip(gold_tags, pred_tags):
        g_ents = get_entities(g_seq)
        p_ents = get_entities(p_seq)
        
        total_gold_entities += len(g_ents)
        total_pred_entities += len(p_ents)
        correct_entities += len(g_ents & p_ents)
        
    precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0.0
    recall = correct_entities / total_gold_entities if total_gold_entities > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "precision": precision * 100,
        "recall": recall * 100,
        "f1": f1 * 100,
        "gold_count": total_gold_entities,
        "pred_count": total_pred_entities,
        "correct_count": correct_entities
    }
            


### Step 8: Hyperparameter Sweep Master Loop - Grid Search Exploration on Fold 0

This sweep loops through your selected parameter combinations. It constructs independent model configurations, trains them for a short duration (e.g., **1 epoch**) on **Fold 0**, decodes predictions, extracts scores using exact entity-level validation, and creates a ranked leaderboard.

In [ ]:
import os
import subprocess
import glob
import numpy as np

# Setups and sweep constraints
setups = ["bioes_word", "bioes_syllable"]
num_epochs = 1  # Short-epoch sweep to evaluate initial training trajectory
fold_to_use = 0

# Specify parameter variants to explore
param_variants = [
    {"name": "ADAM_lr0.001_drop0.3_h200", "optimizer": "ADAM", "learning_rate": 0.001, "dropout": 0.3, "hidden_dim": 200, "batch_size": 10},
    {"name": "ADAM_lr0.002_drop0.4_h150", "optimizer": "ADAM", "learning_rate": 0.002, "dropout": 0.4, "hidden_dim": 150, "batch_size": 10},
    {"name": "SGD_lr0.015_drop0.5_h200",   "optimizer": "SGD",  "learning_rate": 0.015, "dropout": 0.5, "hidden_dim": 200, "batch_size": 16},
    {"name": "ADAM_lr0.0005_drop0.2_h150", "optimizer": "ADAM", "learning_rate": 0.0005, "dropout": 0.2, "hidden_dim": 150, "batch_size": 10},
]

sweep_results = {}
models_root = '/NCRFpp/models'
output_root = '/NCRFpp/output'

for setup in setups:
    sweep_results[setup] = []
    print("="*80)
    print(f"HYPERPARAMETER SWEEP: {setup}")
    print("="*80)

    for variant in param_variants:
        var_name = variant["name"]
        config_name = f"sweep_{setup}_{var_name}"
        pred_file = os.path.join(output_root, f"{config_name}.test.out")

        # --- 1. Dynamic Train Config & Run ---
        create_sweep_config(setup, fold=fold_to_use, with_emb=True, params=variant, config_name=config_name, epochs=num_epochs)
        
        print(f"\n[Sweep Variant: {var_name}] Training...")
        train_config = f"/NCRFpp/{config_name}.train.config"
        train_cmd = ["python", "-u", "/NCRFpp/main.py", "--config", train_config]

        process = subprocess.Popen(
            train_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )

        for line in process.stdout:
            line_str = line.strip()
            if any(kw in line_str for kw in ["Epoch:", "epoch:", "loss:", "Loss:", "f1:", "F1:"]):
                print(f"    [NCRF++] {line_str}")

        process.wait()
        if process.returncode != 0:
            print(f"  Warning: Training failed for variant {var_name}")
            continue

        # --- 2. Decode Predictions ---
        model_files = sorted(glob.glob(os.path.join(models_root, f"{config_name}.*.model")))
        dset_path = os.path.join(models_root, f"{config_name}.dset")

        if not model_files or not os.path.exists(dset_path):
            print(f"  Error: No model checkpoint found for {config_name}")
            continue

        best_model_path = model_files[-1]
        create_fold_decode_config(config_name, setup, fold_to_use, best_model_path, dset_path)

        print(f"  [Sweep Variant: {var_name}] Decoding predictions...")
        decode_config = f"/NCRFpp/{config_name}.decode.config"
        decode_cmd = ["python", "-u", "/NCRFpp/main.py", "--config", decode_config]
        subprocess.run(decode_cmd, stdout=subprocess.DEVNULL, check=True)

        # --- 3. Parse Performance on Test Set ---
        metrics = evaluate_predictions(pred_file)
        if metrics:
            metrics["variant"] = var_name
            metrics["params"] = variant
            print(f"  -> Success: F1-Score: {metrics['f1']:.2f}% (Precision: {metrics['precision']:.2f}%, Recall: {metrics['recall']:.2f}%)")
            sweep_results[setup].append(metrics)
        else:
            print(f"  Error: Output metrics could not be parsed for {config_name}")

# --- 4. Generate Leaderboard Rank Report ---
print("\n" + "="*80)
print("             HYPERPARAMETER LEADERBOARD RANKINGS (FOLD 0 SWEEP)")
print("="*80)

for setup in setups:
    print(f"\nSetup: {setup}")
    print("-" * 65)
    variants_data = sweep_results.get(setup, [])
    if not variants_data:
        print("  * No sweep results available.")
        continue

    ranked = sorted(variants_data, key=lambda x: x["f1"], reverse=True)
    for rank, r in enumerate(ranked, 1):
        p_str = f"P: {r['precision']:5.2f}%"
        r_str = f"R: {r['recall']:5.2f}%"
        f_str = f"F1: {r['f1']:5.2f}%"
        print(f"  #{rank} | {r['variant']:32s} | {f_str} | {p_str} | {r_str}")
    print("-" * 65)
print("="*80)


### Step 8.5: Plot Hyperparameter Search Visualizations

This cell visualizes the comparative F1-scores of your hyperparameter sweep. It displays horizontal ranked bar charts side-by-side for each configuration, highlighting the top-performing combination and its exact metric value.

In [ ]:
import matplotlib.pyplot as plt

def plot_sweep_leaderboard():
    if not sweep_results:
        print("No sweep results found in memory. Please execute the grid sweep master loop first!")
        return

    for setup in setups:
        variants_data = sweep_results.get(setup, [])
        if not variants_data:
            print(f"No sweep data to plot for setup {setup}")
            continue

        # Sort variants from lowest to highest for a horizontal ranking chart
        ranked = sorted(variants_data, key=lambda x: x["f1"])
        names = [r["variant"] for r in ranked]
        f1_scores = [r["f1"] for r in ranked]

        plt.figure(figsize=(11, 4))
        
        # Plot bars (highlighting the top performer with a bright green)
        colors = ['#1f77b4'] * (len(ranked) - 1) + ['#2ca02c']
        bars = plt.barh(names, f1_scores, color=colors, edgecolor='none', height=0.6)
        
        plt.xlabel("Test Set F1-Score (%)", fontsize=10, fontweight='bold')
        plt.title(f"{setup}: Hyperparameter Variant Rankings on Fold 0", fontsize=12, fontweight='bold', pad=15)
        plt.grid(True, axis='x', linestyle="--", alpha=0.5)

        # Annotate each horizontal bar with its exact score
        for bar in bars:
            width = bar.get_width()
            plt.text(width + 0.5, bar.get_y() + bar.get_height()/2, f'{width:.2f}%', 
                     va='center', ha='left', fontweight='bold', fontsize=9, color='#333333')

        # Add visual padding
        plt.xlim(0, max(f1_scores) + 12)
        plt.gca().spines['top'].set_visible(False)
        plt.gca().spines['right'].set_visible(False)
        plt.gca().spines['left'].set_color('#cccccc')
        plt.gca().spines['bottom'].set_color('#cccccc')
        
        plt.tight_layout()
        plt.show()

plot_sweep_leaderboard()


### Step 9: Backup All Experiments to Google Drive


In [ ]:
from datetime import datetime

# Define Google Drive export path
drive_export_path = '/content/drive/My Drive/MyAgriNER/'
ncrfpp_backup_path = os.path.join(drive_export_path, 'NCRFpp_backup')
os.makedirs(ncrfpp_backup_path, exist_ok=True)

src_ncrfpp_dir = '/NCRFpp'
dst_ncrfpp_dir = os.path.join(ncrfpp_backup_path, datetime.now().strftime('%Y%m%d_%H%M%S'))

print(f'Starting backup of {src_ncrfpp_dir} to {dst_ncrfpp_dir}...')

if os.path.exists(src_ncrfpp_dir):
    if os.path.exists(dst_ncrfpp_dir):
        shutil.rmtree(dst_ncrfpp_dir)
    shutil.copytree(src_ncrfpp_dir, dst_ncrfpp_dir)
    print(f'\nBackup complete! The entire directory is saved in {dst_ncrfpp_dir}.')
else:
    print(f'Error: Source directory {src_ncrfpp_dir} does not exist.')
